In [1]:
import cv2
import numpy as np
import os
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            img = cv2.resize(img, (128, 128))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

In [2]:
barren_images, barren_labels = load_images_from_folder('barren', 0)
lush_images, lush_labels = load_images_from_folder('lush', 1)

X = np.concatenate((barren_images, lush_images), axis=0)
y = np.concatenate((barren_labels, lush_labels), axis=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification: barren or lush
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 126, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 63, 63, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 61, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 30, 30, 64)       0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 57600)             0         
                                                                 
 dense (Dense)               (None, 128)               7

2024-11-10 21:42:37.768695: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))


Epoch 1/10
1/1 [==============================] - 2s 2s/step - loss: 6.7592 - accuracy: 0.5217 - val_loss: 439.9221 - val_accuracy: 0.5000
Epoch 2/10
1/1 [==============================] - 0s 449ms/step - loss: 592.7841 - accuracy: 0.4783 - val_loss: 289.3877 - val_accuracy: 0.5000
Epoch 3/10
1/1 [==============================] - 0s 385ms/step - loss: 400.3024 - accuracy: 0.4783 - val_loss: 108.0848 - val_accuracy: 0.5000
Epoch 4/10
1/1 [==============================] - 0s 386ms/step - loss: 160.3722 - accuracy: 0.4783 - val_loss: 2.2155e-20 - val_accuracy: 1.0000
Epoch 5/10
1/1 [==============================] - 0s 388ms/step - loss: 2.6969 - accuracy: 0.9565 - val_loss: 126.3593 - val_accuracy: 0.5000
Epoch 6/10
1/1 [==============================] - 0s 381ms/step - loss: 135.1987 - accuracy: 0.5217 - val_loss: 72.3242 - val_accuracy: 0.5000
Epoch 7/10
1/1 [==============================] - 0s 385ms/step - loss: 82.3943 - accuracy: 0.5217 - val_loss: 2.0752e-11 - val_accuracy: 1.00

In [ ]:
import imutils

camera = cv2.VideoCapture(0)
while True:
    ret, frame = camera.read()
    if not ret:
        break
    frame_resized = cv2.resize(frame, (128, 128))
    frame_expanded = np.expand_dims(frame_resized, axis=0)
    prediction = model.predict(frame_expanded)
    
    if prediction[0] < 0.5:  # Adjust threshold if necessary
        label = "Barren Land"
        color = (0, 0, 255)  # Red for barren
    else:
        label = "Lush Land"
        color = (0, 255, 0)  # Green for lush

    cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.imshow("Land Detector", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()
